# MAGPIE Sanity Check on BABE

This notebook provides a pipeline sanity check for the MAGPIE model using the BABE dataset. BABE is an English-language dataset that was used in the development and fine-tuning of MAGPIE, so the results are treated as a check of the analysis pipeline rather than as independent evidence of model generalisation.

The main purpose is to verify that MAGPIE can produce sentence-level predictions that are consistently aggregated into article-level scores. The analysis also checks whether the expected difference between left- and right-labelled articles is reproduced on this training-adjacent dataset.

**Dataset:** BABE (Spinde et al., 2021), sourced from the Media Bias Group GitHub repository  
**Model:** `mediabiasgroup/magpie-babe-ft-xlm`

## 1. Setup

The notebook starts by installing the packages required for the analysis. The MAGPIE checkpoint used here is the same model used in the main German analysis so that the scoring procedure remains consistent across the two notebooks.

### Run order

Run the notebook **from top to bottom**. The sampling, MAGPIE scoring, article-level aggregation, and statistical comparisons are all based on the same selected article sample.

In [ ]:
!pip install transformers pandas scipy statsmodels -q

## 2. Load the data and classify outlets

BABE is provided at the sentence level. The two label files are combined first, and outlet names are standardised so that different capitalisations of the same outlet are treated consistently.

For this sanity check, outlet information is used to assign left- and right-labelled groups. The resulting comparison therefore reflects the outlet-based labels provided through the dataset rather than independent article-level political annotations.

In [2]:
import pandas as pd

babe_sg1 = pd.read_excel("final_labels_SG1.xlsx")
babe_sg2 = pd.read_excel("final_labels_SG2.xlsx")
babe = pd.concat([babe_sg1, babe_sg2], ignore_index=True)

print("Combined dataset shape:", babe.shape)


Combined dataset shape: (5374, 8)


In [3]:
# Standardise outlet names before assigning the left/right labels.
babe["outlet_clean"] = babe["outlet"].str.lower().str.strip()

# Assign outlets to the left/right categories used in this sanity check.
right_outlets = ["breitbart", "federalist", "fox news", "fox-news", "daily stormer"]
left_outlets = ["alternet", "msnbc", "huffpost", "daily beast", "new york times"]

def classify_outlet(outlet: str) -> str:
    if outlet in right_outlets:
        return "right"
    elif outlet in left_outlets:
        return "left"
    return "center"

babe["stance"] = babe["outlet_clean"].apply(classify_outlet)
print("Stance counts:")
print(babe["stance"].value_counts())


Stance counts:
stance
right     2199
left      1900
center    1275
Name: count, dtype: int64


## 3. Data quality checks

Before drawing the sample, a small set of data quality checks is carried out. Since the article text is used as the input to MAGPIE, records with missing or empty text cannot be included in the analysis. Missing `news_link` values do not affect sentence-level scoring, but they prevent the corresponding sentences from being reliably linked to an article. These records are therefore excluded when constructing the article-level sample.

In [4]:
print("Missing text:", babe["text"].isna().sum())
print("Missing news links:", babe["news_link"].isna().sum())
print("Empty text:", babe["text"].fillna("").str.strip().eq("").sum())
print("Missing outlet:", babe["outlet"].isna().sum())
print("Duplicate rows:", babe.duplicated().sum())

assert babe["text"].notna().all()
assert babe["text"].str.strip().ne("").all()
assert babe["outlet"].notna().all()


Missing text: 0
Missing news links: 60
Empty text: 0
Missing outlet: 0
Duplicate rows: 0


## 4. Build a balanced article sample

BABE is provided at the sentence level, but the analysis is performed at the article level. For this reason, the sampling is done at the article level rather than by selecting individual sentences.

The same number of left- and right-labelled articles is selected using a fixed random seed. Once the articles are selected, all available sentences from those articles are retained for MAGPIE scoring.

In [5]:
# Select complete articles before scoring their sentences.

article_info = (
    babe[babe["news_link"].notna()]
    .groupby("news_link")
    .agg(
        stance=("stance", "first"),
        stance_labels=("stance", "nunique"),
        num_sentences=("text", "count")
    )
    .reset_index()
)

assert article_info["stance_labels"].eq(1).all()
article_info = article_info[article_info["stance"].isin(["left", "right"])].copy()

left_articles = article_info[article_info["stance"] == "left"]
right_articles = article_info[article_info["stance"] == "right"]

sample_size = min(len(left_articles), len(right_articles))

left_article_sample = left_articles.sample(
    n=sample_size, random_state=42
)
right_article_sample = right_articles.sample(
    n=sample_size, random_state=42
)

selected_articles = pd.concat(
    [left_article_sample, right_article_sample]
).sample(
    frac=1, random_state=42
).reset_index(drop=True)

selected_links = selected_articles["news_link"].tolist()

babe_sample = babe[
    babe["news_link"].isin(selected_links)
].copy()

print("Selected articles:", len(selected_articles))
print("\nArticle stance counts:")
print(selected_articles["stance"].value_counts())
print("\nSentence-level sample shape:", babe_sample.shape)
print("\nSentences by stance:")
print(babe_sample["stance"].value_counts())

assert len(selected_articles) == sample_size * 2
assert selected_articles["news_link"].nunique() == len(selected_articles)
assert selected_articles["stance"].value_counts().to_dict() == {
    "left": sample_size,
    "right": sample_size
}


Selected articles: 1306

Article stance counts:
stance
right    653
left     653
Name: count, dtype: int64

Sentence-level sample shape: (3559, 10)

Sentences by stance:
stance
left     1890
right    1669
Name: count, dtype: int64


## 5. Score every sentence with MAGPIE

Since BABE is provided at the sentence level, the selected sentences can be scored directly with MAGPIE. The model returns a probability for the `biased` class, which is stored as `biased_prob`. These sentence-level probabilities are then used to calculate the article-level scores.

In [5]:
%pip install -q --upgrade --force-reinstall torch==2.11.0 torchvision==0.26.0 torchaudio==2.11.0 --index-url https://download.pytorch.org/whl/cu128

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 657.9/657.9 MB 64.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 287.2/287.2 MB 93.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.8/296.8 MB 4.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.1/139.1 MB 113.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 68.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 180.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 77.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 233.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 209.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 135.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.5/267.5 MB 97.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 288.2/288.2 MB 99.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━

In [6]:
import torch
from transformers import pipeline

MODEL_NAME = "mediabiasgroup/magpie-babe-ft"
RANDOM_SEED = 42

assert torch.cuda.is_available(), "CUDA is not available. Check the Colab GPU runtime."
print("GPU:", torch.cuda.get_device_name(0))

babe_classifier = pipeline(
    "text-classification",
    model=MODEL_NAME,
    device=0
)

texts = babe_sample["text"].tolist()
results = babe_classifier(texts, truncation=True, batch_size=32)

babe_sample["label"] = [r["label"] for r in results]
babe_sample["score"] = [r["score"] for r in results]
babe_sample["biased_prob"] = babe_sample.apply(
    lambda row: row["score"] if row["label"] == "biased" else 1 - row["score"], axis=1
)

print("Model device: GPU")
print("Scored sentences:", len(babe_sample))


GPU: Tesla T4


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Model device: GPU
Scored sentences: 3559


## 6. Group sentences back into articles

The selected sentences are linked to their original articles using `news_link`. The sentence-level MAGPIE probabilities are then aggregated within each article to obtain the average, maximum, and median bias scores.

In [7]:
babe_articles = babe_sample.groupby("news_link").agg(
    avg_bias_score=("biased_prob", "mean"),
    max_bias_score=("biased_prob", "max"),
    median_bias_score=("biased_prob", "median"),
    num_sentences=("biased_prob", "count"),
    stance=("stance", "first")
).reset_index()

print("Article groups:", len(babe_articles))
print("\nArticles by stance:")
print(babe_articles["stance"].value_counts())

assert babe_articles["news_link"].nunique() == len(babe_articles)
assert babe_articles["stance"].isin(["left", "right"]).all()


Article groups: 1306

Articles by stance:
stance
right    653
left     653
Name: count, dtype: int64


## 7. Statistical tests

The three article-level pooling measures are compared between the left- and right-labelled groups using Mann–Whitney U tests. These comparisons are used to check whether the expected left/right separation is reproduced in the BABE data. Since BABE is training-adjacent to MAGPIE, the results are interpreted as a pipeline sanity check rather than independent evidence of model generalisation.

In [8]:
from scipy.stats import mannwhitneyu

left_avg = babe_articles[babe_articles["stance"] == "left"]["avg_bias_score"]
right_avg = babe_articles[babe_articles["stance"] == "right"]["avg_bias_score"]
left_max = babe_articles[babe_articles["stance"] == "left"]["max_bias_score"]
right_max = babe_articles[babe_articles["stance"] == "right"]["max_bias_score"]
left_median = babe_articles[babe_articles["stance"] == "left"]["median_bias_score"]
right_median = babe_articles[babe_articles["stance"] == "right"]["median_bias_score"]

stat_avg, p_avg = mannwhitneyu(left_avg, right_avg, alternative="two-sided")
stat_max, p_max = mannwhitneyu(left_max, right_max, alternative="two-sided")
stat_median, p_median = mannwhitneyu(left_median, right_median, alternative="two-sided")

print("AVERAGE POOLING")
print(f"  Left mean: {left_avg.mean():.4f}")
print(f"  Right mean: {right_avg.mean():.4f}")
print(f"  P-value: {p_avg:.4f}")
print()
print("MAX POOLING")
print(f"  Left mean: {left_max.mean():.4f}")
print(f"  Right mean: {right_max.mean():.4f}")
print(f"  P-value: {p_max:.4f}")
print()
print("MEDIAN POOLING")
print(f"  Left mean: {left_median.mean():.4f}")
print(f"  Right mean: {right_median.mean():.4f}")
print(f"  P-value: {p_median:.4f}")


AVERAGE POOLING
  Left mean: 0.5823
  Right mean: 0.5230
  P-value: 0.0001

MAX POOLING
  Left mean: 0.6528
  Right mean: 0.5853
  P-value: 0.0000

MEDIAN POOLING
  Left mean: 0.5883
  Right mean: 0.5273
  P-value: 0.0000


## 8. Checking for a length confound

Article length is checked because the number of sentences in an article may influence the pooled MAGPIE scores. The correlations below are used as a simple diagnostic to see how sentence count relates to the three article-level measures.

In [9]:
print(babe_articles.groupby("stance")["num_sentences"].describe())
print()
print(
    babe_articles[
        ["num_sentences", "avg_bias_score", "max_bias_score", "median_bias_score"]
    ].corr()
)


        count      mean       std  min  25%  50%  75%   max
stance                                                     
left    653.0  2.894334  2.318805  1.0  2.0  2.0  4.0  18.0
right   653.0  2.555896  1.968778  1.0  1.0  2.0  3.0  15.0

                   num_sentences  avg_bias_score  max_bias_score  \
num_sentences           1.000000        0.157384        0.288794   
avg_bias_score          0.157384        1.000000        0.940685   
max_bias_score          0.288794        0.940685        1.000000   
median_bias_score       0.174753        0.988863        0.924594   

                   median_bias_score  
num_sentences               0.174753  
avg_bias_score              0.988863  
max_bias_score              0.924594  
median_bias_score           1.000000  


## 9. Correcting for multiple tests

Three pooling strategies are compared in the BABE sanity check: average, maximum, and median pooling. A Bonferroni correction is applied across these three tests to account for multiple comparisons.

In [10]:
# Three pooling comparisons are included in this sanity check.
num_tests = 3
corrected_alpha = 0.05 / num_tests
print(f"Corrected significance threshold: {corrected_alpha:.4f}")

results = {
    "Average pooling": p_avg,
    "Max pooling": p_max,
    "Median pooling": p_median
}

for method, p in results.items():
    verdict = "SIGNIFICANT" if p < corrected_alpha else "not significant"
    print(f"{method}: p={p:.4f} -> {verdict} (corrected threshold: {corrected_alpha:.4f})")


Corrected significance threshold: 0.0167
Average pooling: p=0.0001 -> SIGNIFICANT (corrected threshold: 0.0167)
Max pooling: p=0.0000 -> SIGNIFICANT (corrected threshold: 0.0167)
Median pooling: p=0.0000 -> SIGNIFICANT (corrected threshold: 0.0167)


## 10. Interpretation

The BABE results are used to check whether the MAGPIE scoring and article-level aggregation pipeline produces a consistent left/right difference on training-adjacent data. Since MAGPIE was fine-tuned on BABE, these results are not treated as independent evidence of generalisation or as independent validation of political bias detection.

The main cross-lingual evaluation is conducted on the German dataset in Notebook 1. The independent English comparison is presented separately in the SemEval analysis.